# Agentic Review Toy Lab (Jupyter)

A self-contained notebook for students to experiment with an **agentic code review** setup: a coordinator + static checks + an LLM specialist reviewer.

This notebook is intentionally **toy and safe**: it uses **small synthetic diffs** embedded below. It does **not** read or send any repository code unless you paste it in.

## What You Will Do

1. Set your own API key (OpenRouter / OpenAI / Groq).
2. Select a model.
3. Run the toy review agent and inspect outputs.
4. Cause **false positives** on purpose, then reduce them by improving the prompt.
5. Score results using **TP / TN / FP / FN** (plus precision/recall/F1).

Cost note: the LLM calls cost money. Start with a small, cheap model.

## Preconditions

- Python 3.10+ recommended
- Network access to your chosen provider
- No repo required (this notebook is standalone)

In [ ]:
from __future__ import annotations

import json
import os
import re
import time
from dataclasses import dataclass
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

import sys
print('Python:', sys.version.split()[0])


## 1) Set Your API Key

Set one of these environment variables:
- `OPENROUTER_API_KEY`
- `OPENAI_API_KEY`
- `GROQ_API_KEY`

The next cell sets keys **in memory** for this notebook session using `getpass()` (no echo).
Do not commit keys into notebooks.

In [ ]:
import getpass

def set_key(env_name: str) -> None:
    existing = (os.environ.get(env_name) or '').strip()
    if existing:
        print(f'{env_name} already set (len={len(existing)}).')
        return
    value = getpass.getpass(f'Enter {env_name} (leave blank to skip): ').strip()
    if not value:
        print(f'Skipped {env_name}.')
        return
    os.environ[env_name] = value
    print(f'Set {env_name} (len={len(value)}).')

set_key('OPENROUTER_API_KEY')
set_key('OPENAI_API_KEY')
set_key('GROQ_API_KEY')


## 2) Choose Provider + Model

Provider options:
- `openrouter`: model ids look like `openai/gpt-4o-mini`
- `openai`: model ids look like `gpt-4o-mini`
- `groq`: model ids look like `llama-3.1-8b-instant`

Default here is `openrouter` + `openai/gpt-4o-mini` (cheap/fast).

In [ ]:
PROVIDER = 'openrouter'  # 'openrouter' | 'openai' | 'groq'

DEFAULT_MODELS = {
    'openrouter': 'openai/gpt-4o-mini',
    'openai': 'gpt-4o-mini',
    'groq': 'llama-3.1-8b-instant',
}

MODEL = DEFAULT_MODELS[PROVIDER]

print('PROVIDER =', PROVIDER)
print('MODEL    =', MODEL)


## 3) Minimal OpenAI-Compatible HTTP Client

This client calls `POST /chat/completions` against an OpenAI-compatible API.
It works for OpenRouter, OpenAI, and Groq.

In [ ]:
class ProviderError(RuntimeError):
    pass

def _normalize_message_content(content: object) -> str:
    if isinstance(content, str):
        return content.strip()
    # Some providers can return list content segments.
    if isinstance(content, list):
        parts: list[str] = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and isinstance(item.get('text'), str):
                parts.append(item['text'])
        joined = ''.join(parts).strip()
        if joined:
            return joined
    raise ProviderError(f'Unsupported message content type: {type(content).__name__}')

@dataclass
class ChatCompletionsHTTPClient:
    base_url: str
    api_key: str
    timeout_seconds: int = 45
    max_retries: int = 2
    extra_headers: dict[str, str] | None = None

    def complete(self, *, model: str, system_prompt: str, user_prompt: str) -> str:
        if not self.api_key.strip():
            raise ProviderError('Missing API key for selected provider.')

        payload = {
            'model': model,
            'temperature': 0,
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
        }
        data = json.dumps(payload).encode('utf-8')
        url = f"{self.base_url.rstrip('/')}/chat/completions"

        request = Request(url, data=data, method='POST')
        request.add_header('Content-Type', 'application/json')
        request.add_header('Accept', 'application/json')
        request.add_header('Authorization', f'Bearer {self.api_key}')
        for k, v in (self.extra_headers or {}).items():
            request.add_header(k, v)

        last_error: Exception | None = None
        for attempt in range(self.max_retries + 1):
            try:
                with urlopen(request, timeout=self.timeout_seconds) as resp:
                    raw = resp.read()
                body = json.loads(raw.decode('utf-8'))
                if not isinstance(body, dict):
                    raise ProviderError('Provider returned non-object JSON.')
                choices = body.get('choices')
                if not isinstance(choices, list) or not choices:
                    raise ProviderError('Provider response missing choices[].')
                first = choices[0]
                if not isinstance(first, dict):
                    raise ProviderError('Provider response choices[0] is not an object.')
                message = first.get('message')
                if not isinstance(message, dict):
                    raise ProviderError('Provider response missing message object.')
                return _normalize_message_content(message.get('content'))
            except HTTPError as exc:
                last_error = exc
                retryable = exc.code in {408, 425, 429, 500, 502, 503, 504}
                if attempt >= self.max_retries or not retryable:
                    raise ProviderError(f'HTTPError from provider: {exc}') from exc
                time.sleep(min(2 ** attempt, 10))
            except (URLError, TimeoutError, json.JSONDecodeError, UnicodeDecodeError) as exc:
                last_error = exc
                if attempt >= self.max_retries:
                    raise ProviderError(f'Provider request failed: {exc}') from exc
                time.sleep(min(2 ** attempt, 10))

        raise ProviderError('Provider request failed without a usable error.') from last_error


def build_client(provider: str) -> ChatCompletionsHTTPClient:
    provider = provider.strip().lower()
    if provider == 'openrouter':
        key = (os.environ.get('OPENROUTER_API_KEY') or '').strip()
        return ChatCompletionsHTTPClient(
            base_url='https://openrouter.ai/api/v1',
            api_key=key,
            extra_headers={
                'X-Title': 'Agentic Review Toy Lab',
            },
        )
    if provider == 'openai':
        key = (os.environ.get('OPENAI_API_KEY') or '').strip()
        return ChatCompletionsHTTPClient(base_url='https://api.openai.com/v1', api_key=key)
    if provider == 'groq':
        key = (os.environ.get('GROQ_API_KEY') or '').strip()
        return ChatCompletionsHTTPClient(base_url='https://api.groq.com/openai/v1', api_key=key)
    raise ValueError(f'Unknown provider: {provider}')

CLIENT: ChatCompletionsHTTPClient | None = None
try:
    CLIENT = build_client(PROVIDER)
    if not CLIENT.api_key:
        print('Client created, but API key is missing. Set a key above to run LLM cells.')
    else:
        print('Client ready for provider:', PROVIDER)
except Exception as exc:
    print('Client setup failed:', exc)


## 4) The Specialist JSON Contract

Our toy specialist must return valid JSON with this shape:

```json
{
  "findings": [
    {
      "title": "...",
      "severity": "critical|high|moderate|low",
      "confidence": "high|medium|low",
      "category": "correctness|tests|security",
      "file": "path/to/file.py",
      "line_start": 1,
      "line_end": 1,
      "evidence": "quote changed + / - line(s)",
      "impact": "concrete failure scenario",
      "suggested_action": "specific fix",
      "blocking_recommendation": true
    }
  ],
  "uncertain_risks": [
    {
      "risk": "...",
      "reason_uncertain": "...",
      "suggested_verification": "..."
    }
  ],
  "note": "optional"
}
```

We also add robust parsing to handle common model mistakes.

In [ ]:
BASE_SYSTEM_PROMPT = '''
You are a code review specialist.
Review ONLY what is present in the provided diff.
Do not invent context.

Rules:
- You MAY return zero findings.
- Every finding MUST cite concrete evidence from a changed diff line starting with + or -.
- If you are unsure, put it in uncertain_risks, not findings.
- Return valid JSON only (no markdown, no commentary).

Cap findings at 3.
'''

CORRECTNESS_SPECIALIST_PROMPT = '''
Focus on correctness/regression issues in changed code (not style).
Look for:
- exceptions on edge cases
- incorrect indexing
- missing error handling
- mutable default arguments
- security footguns like os.system/eval

Return JSON in the required shape.
'''

print('Prompts ready.')


## 5) The 8-Example Mini-Benchmark (Ground Truth Labels)

We will classify each diff as:
- **issue present** (should produce at least one finding)
- **no issue** (should produce zero findings)

Then compute TP/TN/FP/FN.

In [ ]:
BENCHMARK: list[dict[str, Any]] = [
    {
        'id': 'P1',
        'expected_has_issue': True,
        'desc': 'Mutable default argument (shared state bug).',
        'diff': '''
diff --git a/app/utils.py b/app/utils.py
index 1111111..2222222 100644
--- a/app/utils.py
+++ b/app/utils.py
@@ -1,5 +1,4 @@
-def add_tag(tag: str, tags: list[str] | None = None) -> list[str]:
-    tags = [] if tags is None else tags
-    tags.append(tag)
-    return tags
+def add_tag(tag: str, tags: list[str] = []) -> list[str]:
+    tags.append(tag)
+    return tags
'''
    },
    {
        'id': 'P2',
        'expected_has_issue': True,
        'desc': 'Removes ValueError handling (crash on invalid input).',
        'diff': '''
diff --git a/app/parse.py b/app/parse.py
index 3333333..4444444 100644
--- a/app/parse.py
+++ b/app/parse.py
@@ -10,8 +10,5 @@
 def parse_int(value: str) -> int:
-    try:
-        return int(value)
-    except ValueError:
-        return 0
+    return int(value)
'''
    },
    {
        'id': 'P3',
        'expected_has_issue': True,
        'desc': 'Off-by-one indexing bug (IndexError risk).',
        'diff': '''
diff --git a/app/math.py b/app/math.py
index 5555555..6666666 100644
--- a/app/math.py
+++ b/app/math.py
@@ -20,6 +20,6 @@
 def sum_items(items: list[int]) -> int:
     total = 0
     for i in range(len(items)):
-        total += items[i]
+        total += items[i + 1]
     return total
'''
    },
    {
        'id': 'P4',
        'expected_has_issue': True,
        'desc': 'Command injection risk via os.system(user_input).',
        'diff': '''
diff --git a/app/exec.py b/app/exec.py
index 7777777..8888888 100644
--- a/app/exec.py
+++ b/app/exec.py
@@ -1,7 +1,9 @@
+import os
+
 def run_user_command(cmd: str) -> int:
-    return 0
+    return os.system(cmd)
'''
    },
    {
        'id': 'N1',
        'expected_has_issue': False,
        'desc': 'Comment-only change.',
        'diff': '''
diff --git a/app/notes.py b/app/notes.py
index 9999999..aaaaaaaa 100644
--- a/app/notes.py
+++ b/app/notes.py
@@ -1,4 +1,4 @@
-# Utility helpers
+# Utility helpers (clarify intent)
 def noop() -> None:
     return
'''
    },
    {
        'id': 'N2',
        'expected_has_issue': False,
        'desc': 'Pure rename (no logic change).',
        'diff': '''
diff --git a/app/core.py b/app/core.py
index bbbbbbb..ccccccc 100644
--- a/app/core.py
+++ b/app/core.py
@@ -40,5 +40,5 @@
-def compute(x: int) -> int:
-    return x + 1
+def compute_next(x: int) -> int:
+    return x + 1
'''
    },
    {
        'id': 'N3',
        'expected_has_issue': False,
        'desc': 'Type annotation only.',
        'diff': '''
diff --git a/app/types.py b/app/types.py
index ddddddd..eeeeeee 100644
--- a/app/types.py
+++ b/app/types.py
@@ -50,3 +50,3 @@
-def identity(x):
+def identity(x: int) -> int:
     return x
'''
    },
    {
        'id': 'N4',
        'expected_has_issue': False,
        'desc': 'Adds debug print (assume acceptable for this toy).',
        'diff': '''
diff --git a/app/logging.py b/app/logging.py
index fffffff..0000000 100644
--- a/app/logging.py
+++ b/app/logging.py
@@ -60,4 +60,6 @@
 def do_work(x: int) -> int:
+    print(f'working on {x}')
     return x * 2
'''
    },
]

print('Loaded benchmark examples:', len(BENCHMARK))


## 6) Toy Diff Parsing + Toy Static Checks

We parse unified diffs just enough to:
- extract file paths
- track line numbers for added lines

Then we run a few deterministic (very naive) static checks.

In [ ]:
HUNK_RE = re.compile(r'^@@ -\d+(?:,\d+)? \+(?P<start>\d+)(?:,(?P<count>\d+))? @@')

@dataclass(frozen=True)
class AddedLine:
    file: str
    line_no: int
    text: str

def parse_unified_diff(diff_text: str) -> dict[str, Any]:
    current_file: str | None = None
    new_line: int | None = None
    added: list[AddedLine] = []

    for raw in diff_text.splitlines():
        line = raw.rstrip('
')
        if line.startswith('+++ '):
            # +++ b/path
            path = line[4:].strip()
            if path.startswith('b/'):
                path = path[2:]
            current_file = path
            continue

        m = HUNK_RE.match(line)
        if m:
            new_line = int(m.group('start'))
            continue

        if current_file is None or new_line is None:
            continue
        if line.startswith('+++') or line.startswith('---'):
            continue
        if line.startswith(chr(92)):  # backslash-prefixed diff metadata
            continue

        if line.startswith('+') and not line.startswith('+++'):
            added.append(AddedLine(file=current_file, line_no=new_line, text=line[1:]))
            new_line += 1
        elif line.startswith('-') and not line.startswith('---'):
            # deletion: does not advance new_line
            pass
        else:
            # context
            new_line += 1

    files = sorted({a.file for a in added})
    return {'files': files, 'added_lines': added, 'raw': diff_text}

def static_checks(parsed: dict[str, Any]) -> list[dict[str, Any]]:
    findings: list[dict[str, Any]] = []
    for a in parsed['added_lines']:
        t = a.text
        if 'os.system(' in t or 'eval(' in t:
            findings.append({
                'title': 'Dangerous dynamic execution',
                'severity': 'high',
                'confidence': 'high',
                'category': 'security',
                'file': a.file,
                'line_start': a.line_no,
                'line_end': a.line_no,
                'evidence': f'+{t.strip()}',
                'impact': 'User-controlled input could execute arbitrary commands/code.',
                'suggested_action': 'Avoid os.system/eval on untrusted input; use safe APIs and validation.',
                'blocking_recommendation': True,
                'source': 'static',
            })
        if 'list[str] = []' in t or '=[]' in t.replace(' ', ''):
            findings.append({
                'title': 'Mutable default argument',
                'severity': 'moderate',
                'confidence': 'high',
                'category': 'correctness',
                'file': a.file,
                'line_start': a.line_no,
                'line_end': a.line_no,
                'evidence': f'+{t.strip()}',
                'impact': 'Default list is shared across calls, causing surprising cross-call state.',
                'suggested_action': 'Use None default and create a new list inside the function.',
                'blocking_recommendation': False,
                'source': 'static',
            })
    return findings

example = BENCHMARK[0]
parsed = parse_unified_diff(example['diff'])
print('Parsed files:', parsed['files'])
print('Added lines:', len(parsed['added_lines']))
print('Static findings:', len(static_checks(parsed)))


## 7) Robust JSON Parsing (Common Model Failure Mode)

Models sometimes return extra text, markdown fences, or malformed JSON.
We parse defensively by extracting the first top-level JSON object.

In [ ]:
def load_json_object(raw_response: str) -> dict[str, Any]:
    try:
        payload = json.loads(raw_response)
        return payload if isinstance(payload, dict) else {'findings': [], 'uncertain_risks': [], 'note': 'non-object JSON'}
    except json.JSONDecodeError:
        start = raw_response.find('{')
        end = raw_response.rfind('}')
        if start == -1 or end == -1 or end <= start:
            return {'findings': [], 'uncertain_risks': [], 'note': 'invalid non-JSON response'}
        try:
            payload = json.loads(raw_response[start : end + 1])
            return payload if isinstance(payload, dict) else {'findings': [], 'uncertain_risks': [], 'note': 'non-object JSON'}
        except json.JSONDecodeError:
            return {'findings': [], 'uncertain_risks': [], 'note': 'invalid non-JSON response'}

print(load_json_object('Here you go! findings: [] (not JSON)'))
print(load_json_object('Sure! {"findings": [], "uncertain_risks": [], "note": "ok"} thanks'))


## 8) LLM Specialist Runner

This calls the provider and then validates findings:
- must reference a file that exists in the diff
- must have required fields
- caps findings at 3

If the model returns garbage, you should see the failure show up as `uncertain_risks` instead of crashing.

In [ ]:
ALLOWED_SEVERITIES = {'critical', 'high', 'moderate', 'low'}
ALLOWED_CONFIDENCE = {'high', 'medium', 'low'}
ALLOWED_CATEGORIES = {'correctness', 'tests', 'security'}

def clamp_findings(raw_findings: list[dict[str, Any]], files_in_diff: set[str]) -> tuple[list[dict[str, Any]], list[dict[str, str]]]:
    kept: list[dict[str, Any]] = []
    risks: list[dict[str, str]] = []
    required = {
        'title', 'severity', 'confidence', 'category', 'file',
        'line_start', 'line_end', 'evidence', 'impact', 'suggested_action', 'blocking_recommendation'
    }
    for item in raw_findings[:100]:
        if not isinstance(item, dict) or not required.issubset(item.keys()):
            continue
        path = str(item['file']).strip()
        if path not in files_in_diff:
            risks.append({
                'risk': 'Finding discarded: path not in diff',
                'reason_uncertain': f'Model referenced {path}, but diff files are {sorted(files_in_diff)}',
                'suggested_verification': 'Improve the prompt to use exact diff file paths.'
            })
            continue

        sev = str(item['severity']).strip().lower()
        conf = str(item['confidence']).strip().lower()
        cat = str(item['category']).strip().lower()
        if sev not in ALLOWED_SEVERITIES or conf not in ALLOWED_CONFIDENCE or cat not in ALLOWED_CATEGORIES:
            continue

        kept.append({
            **{k: item[k] for k in required},
            'severity': sev,
            'confidence': conf,
            'category': cat,
            'file': path,
            'source': 'llm',
        })
        if len(kept) >= 3:
            break
    return kept, risks

def run_correctness_specialist(*, diff_text: str, client: ChatCompletionsHTTPClient, model: str, system_prompt: str) -> dict[str, Any]:
    parsed = parse_unified_diff(diff_text)
    files = set(parsed['files'])

    user_prompt = (
        'Review this unified diff. Return JSON only in the required shape.\n\n'
        + diff_text.strip()
    )
    raw = client.complete(model=model, system_prompt=system_prompt, user_prompt=user_prompt)
    data = load_json_object(raw)

    raw_findings = data.get('findings')
    raw_findings_list = raw_findings if isinstance(raw_findings, list) else []
    kept, clamp_risks = clamp_findings(raw_findings_list, files)

    uncertain = data.get('uncertain_risks')
    uncertain_list = uncertain if isinstance(uncertain, list) else []
    normalized_uncertain: list[dict[str, str]] = []
    for r in uncertain_list[:25]:
        if not isinstance(r, dict):
            continue
        if not all(k in r for k in ('risk', 'reason_uncertain', 'suggested_verification')):
            continue
        normalized_uncertain.append({
            'risk': str(r['risk']),
            'reason_uncertain': str(r['reason_uncertain']),
            'suggested_verification': str(r['suggested_verification']),
        })

    normalized_uncertain.extend(clamp_risks)

    return {
        'raw': raw,
        'findings': kept,
        'uncertain_risks': normalized_uncertain,
        'note': str(data.get('note', '')),
        'files': sorted(files),
    }


## 9) Toy Coordinator: Merge Static + LLM, Deduplicate, Verdict

Coordinator responsibilities (toy version):
- run static checks
- run LLM specialist
- dedupe/cap findings
- output a verdict

In [ ]:
SEVERITY_ORDER = {'low': 1, 'moderate': 2, 'high': 3, 'critical': 4}

def dedupe_findings(findings: list[dict[str, Any]]) -> list[dict[str, Any]]:
    best: dict[tuple[str, str, int, str], dict[str, Any]] = {}
    for f in findings:
        key = (str(f.get('category')), str(f.get('file')), int(f.get('line_start', 1)), str(f.get('title', '')).strip().lower())
        existing = best.get(key)
        if existing is None:
            best[key] = f
            continue
        # keep the higher severity on collision
        if SEVERITY_ORDER.get(str(f.get('severity')), 0) > SEVERITY_ORDER.get(str(existing.get('severity')), 0):
            best[key] = f
    ordered = sorted(best.values(), key=lambda x: (bool(x.get('blocking_recommendation')), SEVERITY_ORDER.get(str(x.get('severity')), 0)), reverse=True)
    return ordered

def verdict_for(findings: list[dict[str, Any]], uncertain_risks: list[dict[str, str]]) -> str:
    # Fail closed only on explicit blocking findings.
    for f in findings:
        if bool(f.get('blocking_recommendation')) and str(f.get('severity')) in {'high', 'critical'}:
            return 'NEEDS CHANGES'
    # If we got nothing useful but have major uncertainty, discuss.
    if not findings and uncertain_risks:
        return 'DISCUSS'
    return 'LGTM'

def run_agentic_review(*, diff_text: str, client: ChatCompletionsHTTPClient | None, model: str, system_prompt: str, include_static: bool = True) -> dict[str, Any]:
    parsed = parse_unified_diff(diff_text)

    findings: list[dict[str, Any]] = []
    uncertain: list[dict[str, str]] = []

    if include_static:
        findings.extend(static_checks(parsed))

    if client is None or not client.api_key:
        uncertain.append({
            'risk': 'LLM specialist not run',
            'reason_uncertain': 'No API key was configured for the selected provider.',
            'suggested_verification': 'Set an API key and rerun.'
        })
    else:
        specialist = run_correctness_specialist(
            diff_text=diff_text,
            client=client,
            model=model,
            system_prompt=system_prompt,
        )
        findings.extend(specialist['findings'])
        uncertain.extend(specialist['uncertain_risks'])

    findings = dedupe_findings(findings)[:5]
    verdict = verdict_for(findings, uncertain)

    overall_risk = 'low'
    if findings:
        overall_risk = max((str(f.get('severity')) for f in findings), key=lambda s: SEVERITY_ORDER.get(s, 0))

    return {
        'summary': {
            'overall_risk': overall_risk,
            'finding_count': len(findings),
            'uncertain_risk_count': len(uncertain),
        },
        'files': parsed['files'],
        'findings': findings,
        'uncertain_risks': uncertain,
        'verdict': verdict,
    }

def render_review_md(review: dict[str, Any]) -> str:
    lines = []
    lines.append('# Toy Agentic Review')
    lines.append('')
    lines.append('Verdict: `{}`'.format(review['verdict']))
    lines.append('Overall risk: `{}`'.format(review['summary']['overall_risk']))
    lines.append('')
    lines.append('## Files')
    for p in review.get('files', []):
        lines.append('- `{}`'.format(p))
    lines.append('')
    lines.append('## Findings')
    if not review['findings']:
        lines.append('- No findings.')
    for f in review['findings']:
        lines.append('- **[{}][{}]** {} (`{}:{}`)'.format(f['severity'].upper(), f['category'], f['title'], f['file'], f['line_start']))
        lines.append('  Evidence: {}'.format(f['evidence']))
        lines.append('  Impact: {}'.format(f['impact']))
        lines.append('  Suggested action: {}'.format(f['suggested_action']))
    if review['uncertain_risks']:
        lines.append('')
        lines.append('## Uncertain Risks')
        for r in review['uncertain_risks'][:10]:
            lines.append('- {}'.format(r['risk']))
            lines.append('  Why uncertain: {}'.format(r['reason_uncertain']))
            lines.append('  Verify: {}'.format(r['suggested_verification']))
    return '
'.join(lines)

# Demo on one example
demo = run_agentic_review(diff_text=BENCHMARK[0]['diff'], client=CLIENT, model=MODEL, system_prompt=BASE_SYSTEM_PROMPT + '

' + CORRECTNESS_SPECIALIST_PROMPT)
print(render_review_md(demo))


# Prompt Lab: False Positives + TP/TN/FP/FN

We treat the specialist as a binary classifier:
- prediction = `has_issue` if it returns >= 1 finding
- otherwise `no_issue`

Then compute TP/TN/FP/FN and precision/recall/F1.

In [ ]:
def confusion(rows: list[dict[str, Any]]) -> dict[str, float]:
    tp = tn = fp = fn = 0
    for r in rows:
        y = bool(r['expected_has_issue'])
        yhat = bool(r['predicted_has_issue'])
        if y and yhat:
            tp += 1
        elif (not y) and (not yhat):
            tn += 1
        elif (not y) and yhat:
            fp += 1
        else:
            fn += 1
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'precision': precision, 'recall': recall, 'f1': f1,
    }

def run_benchmark(*, system_prompt: str, include_static: bool = False) -> tuple[list[dict[str, Any]], dict[str, float]]:
    rows: list[dict[str, Any]] = []
    for ex in BENCHMARK:
        review = run_agentic_review(
            diff_text=ex['diff'],
            client=CLIENT,
            model=MODEL,
            system_prompt=system_prompt,
            include_static=include_static,
        )
        predicted_has_issue = len(review['findings']) > 0
        rows.append({
            'id': ex['id'],
            'desc': ex['desc'],
            'expected_has_issue': ex['expected_has_issue'],
            'predicted_has_issue': predicted_has_issue,
            'finding_count': len(review['findings']),
            'verdict': review['verdict'],
            'first_finding_title': (review['findings'][0]['title'] if review['findings'] else ''),
        })
    stats = confusion(rows)
    return rows, stats

BASELINE_PROMPT = BASE_SYSTEM_PROMPT + '

' + CORRECTNESS_SPECIALIST_PROMPT
rows, stats = run_benchmark(system_prompt=BASELINE_PROMPT, include_static=False)
print('Baseline confusion:', stats)
for r in rows:
    print(r['id'], 'expected=', r['expected_has_issue'], 'predicted=', r['predicted_has_issue'], 'findings=', r['finding_count'], r['first_finding_title'])


## Force False Positives (Bad Prompt)

We intentionally add a bad instruction: **always return at least one finding**.
This should increase FP and reduce precision.

In [ ]:
BAD_PROMPT = BASE_SYSTEM_PROMPT + '''

LAB INSTRUCTION (bad on purpose): ALWAYS return at least one finding, even if the diff is harmless.
''' + '

' + CORRECTNESS_SPECIALIST_PROMPT
rows_bad, stats_bad = run_benchmark(system_prompt=BAD_PROMPT, include_static=False)
print('Bad-prompt confusion:', stats_bad)


## Prompt Improvement Task (Reduce False Positives)

Edit `IMPROVED_PROMPT` below to reduce FP while keeping TP.

Prompt ideas:
- Explicitly prefer **zero findings** when evidence is weak
- Require quoting `+` / `-` lines in `evidence`
- If evidence is missing, push it to `uncertain_risks`
- Avoid style-only findings

In [ ]:
IMPROVED_PROMPT = BASE_SYSTEM_PROMPT + '''

Additional guardrails (tune these):
- You are allowed to return an empty findings list.
- Only produce a finding if you can quote at least one changed diff line (starting with + or -) in the evidence field.
- If you cannot quote evidence, do NOT guess; add an uncertain_risks entry instead.
- Do not generate findings for naming/formatting-only diffs.
''' + '

' + CORRECTNESS_SPECIALIST_PROMPT

rows_improved, stats_improved = run_benchmark(system_prompt=IMPROVED_PROMPT, include_static=False)
print('Improved-prompt confusion:', stats_improved)


# Other Error Types (Quick Demos)

Common failure modes you should be ready to handle in an agent:
1. **Invalid JSON** from the model
2. **Path hallucination** (model references a file not in diff)

The next cell demonstrates both without making an LLM call.

In [ ]:
# 1) Invalid JSON
print(load_json_object('not json at all'))

# 2) Path hallucination gets converted to an uncertain risk
diff_text = BENCHMARK[0]['diff']
files = set(parse_unified_diff(diff_text)['files'])
fake = {
    'findings': [
        {
            'title': 'Bug',
            'severity': 'high',
            'confidence': 'high',
            'category': 'correctness',
            'file': 'app/NOT_IN_DIFF.py',
            'line_start': 1,
            'line_end': 1,
            'evidence': '+ whatever',
            'impact': '...',
            'suggested_action': '...',
            'blocking_recommendation': True,
        }
    ],
    'uncertain_risks': [],
    'note': ''
}
kept, risks = clamp_findings(fake['findings'], files)
print('Kept findings:', kept)
print('Converted risks:', risks)


# Wrap-Up

Try these next:
- Switch models (cheap vs strong) and compare FP/FN.
- Tighten your prompt until FP drops, then check whether FN increases.
- Change the decision rule: count a prediction as positive only when severity >= `high`.